### 5. Neural networks
##### Explore varies neural networks using keras library.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping



In [5]:
# first use the basic no text data
data = pd.read_csv("../COMP30027_2024_asst2_data/train_select.csv")
test_data = pd.read_csv("../COMP30027_2024_asst2_data/test_dataset_no_text.csv")
# drop id before calculation, add back after
data = data.drop(columns='id')
test_data = test_data.drop(columns='id')

In [6]:
# split the data
X = data.drop(columns=["imdb_score_binned"])
y = data["imdb_score_binned"]



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=715)

# set up scalers
mmscaler = MinMaxScaler()
ssscaler = StandardScaler()


mmX_train = mmscaler.fit_transform(X_train)
mmX_test = mmscaler.transform(X_test)
ssX_train = ssscaler.fit_transform(X_train)
ssX_test = ssscaler.transform(X_test)



In [7]:
# design nn
model = Sequential([
    Dense(32, input_shape=(X_train.shape[1],), activation='relu'),
    Dense(16, activation='sigmoid'),
    Dense(y.nunique(), activation='softmax')
])

# instantiate
model.compile(optimizer='nadam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(ssX_train, y_train, epochs=50, batch_size=32, validation_split=0.1, callbacks=[early_stopping])


# get accuracy
test_loss, test_acc = model.evaluate(ssX_test, y_test)
print("Test Accuracy:" ,test_acc)


Epoch 1/50
68/68 [==============================] - 1s 2ms/step - loss: 1.2825 - accuracy: 0.5648 - val_loss: 1.0757 - val_accuracy: 0.6183
Epoch 2/50
68/68 [==============================] - 0s 910us/step - loss: 1.0318 - accuracy: 0.6226 - val_loss: 0.9657 - val_accuracy: 0.6432
Epoch 3/50
68/68 [==============================] - 0s 910us/step - loss: 0.9697 - accuracy: 0.6327 - val_loss: 0.9266 - val_accuracy: 0.6390
Epoch 4/50
68/68 [==============================] - 0s 925us/step - loss: 0.9404 - accuracy: 0.6438 - val_loss: 0.9019 - val_accuracy: 0.6349
Epoch 5/50
68/68 [==============================] - 0s 940us/step - loss: 0.9181 - accuracy: 0.6415 - val_loss: 0.8807 - val_accuracy: 0.6390
Epoch 6/50
68/68 [==============================] - 0s 910us/step - loss: 0.9001 - accuracy: 0.6466 - val_loss: 0.8610 - val_accuracy: 0.6556
Epoch 7/50
68/68 [==============================] - 0s 925us/step - loss: 0.8832 - accuracy: 0.6489 - val_loss: 0.8454 - val_accuracy: 0.6556
Epoch 8/

#### Kaggle prediction on test data

In [22]:
mmX = mmscaler.fit_transform(X)
ssX = ssscaler.fit_transform(X)
sstest_data = ssscaler.transform(test_data)

history = model.fit(ssX, y, epochs=20, batch_size=32, validation_split=0.2, callbacks=[early_stopping])

Epoch 1/20
76/76 [==============================] - 0s 2ms/step - loss: 0.6604 - accuracy: 0.7316 - val_loss: 0.6746 - val_accuracy: 0.7354
Epoch 2/20
76/76 [==============================] - 0s 1ms/step - loss: 0.6572 - accuracy: 0.7316 - val_loss: 0.6810 - val_accuracy: 0.7338
Epoch 3/20
76/76 [==============================] - 0s 1ms/step - loss: 0.6533 - accuracy: 0.7337 - val_loss: 0.6769 - val_accuracy: 0.7354
Epoch 4/20
76/76 [==============================] - 0s 1ms/step - loss: 0.6510 - accuracy: 0.7312 - val_loss: 0.6862 - val_accuracy: 0.7271
Epoch 5/20
76/76 [==============================] - 0s 1ms/step - loss: 0.6495 - accuracy: 0.7362 - val_loss: 0.6792 - val_accuracy: 0.7371
Epoch 6/20
76/76 [==============================] - 0s 1ms/step - loss: 0.6476 - accuracy: 0.7341 - val_loss: 0.6826 - val_accuracy: 0.7304
Epoch 7/20
76/76 [==============================] - 0s 1ms/step - loss: 0.6446 - accuracy: 0.7349 - val_loss: 0.6807 - val_accuracy: 0.7388
Epoch 8/20
76/76 [==

In [23]:
test_pred = model.predict(test_data)
test_pred = test_pred.argmax(axis=1)
ids = range(1, len(test_pred) + 1)

# save submission file
prediction = pd.DataFrame({'id': ids, 'imdb_score_binned': test_pred})
print(prediction['imdb_score_binned'].value_counts())
prediction.to_csv("../Kaggle_submissions/dense_nn_select_feature.csv", index=False)

24/24 [==============================] - 0s 957us/step
imdb_score_binned
2    724
4     14
3     14
Name: count, dtype: int64


### 5.2

In [7]:
data = pd.read_csv("../COMP30027_2024_asst2_data/train_select.csv")
# drop id before calculation, add back after
data = data.drop(columns='id')

In [8]:
# split the data
X = data.drop(columns=["imdb_score_binned"])
y = data["imdb_score_binned"]

# set up scalers
mmscaler = MinMaxScaler()
ssscaler = StandardScaler()

mmscaler.fit(X)
ssscaler.fit(X)

mmX = mmscaler.transform(X)
ssX = ssscaler.transform(X)

X_train, X_test, y_train, y_test = train_test_split(ssX, y, test_size=0.2, random_state=715)

In [9]:
# design nn
model = Sequential([
    Dense(32, input_shape=(X_train.shape[1],), activation='relu'),
    Dense(16, activation='relu'), 
    Dense(y.nunique(), activation='softmax')
])

# instantiate
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2)

# get accuracy
test_loss, test_acc = model.evaluate(X_test, y_test)  # Make sure to transform X_test with the same scaler used on X_train
print("Test Accuracy:" ,test_acc)


Epoch 1/50
61/61 [==============================] - 1s 3ms/step - loss: 1.4764 - accuracy: 0.5094 - val_loss: 1.2306 - val_accuracy: 0.6299
Epoch 2/50
61/61 [==============================] - 0s 2ms/step - loss: 1.1713 - accuracy: 0.6077 - val_loss: 1.0777 - val_accuracy: 0.6320
Epoch 3/50
61/61 [==============================] - 0s 1ms/step - loss: 1.0551 - accuracy: 0.6160 - val_loss: 0.9939 - val_accuracy: 0.6362
Epoch 4/50
61/61 [==============================] - 0s 1ms/step - loss: 0.9781 - accuracy: 0.6327 - val_loss: 0.9246 - val_accuracy: 0.6466
Epoch 5/50
61/61 [==============================] - 0s 1ms/step - loss: 0.9159 - accuracy: 0.6478 - val_loss: 0.8735 - val_accuracy: 0.6528
Epoch 6/50
61/61 [==============================] - 0s 1ms/step - loss: 0.8618 - accuracy: 0.6602 - val_loss: 0.8367 - val_accuracy: 0.6653
Epoch 7/50
61/61 [==============================] - 0s 1ms/step - loss: 0.8213 - accuracy: 0.6785 - val_loss: 0.7984 - val_accuracy: 0.6778
Epoch 8/50
61/61 [==

In [10]:
test_data = ssscaler.fit_transform(test_data)
test_pred = model.predict(test_data)
test_pred = test_pred.argmax(axis=1)
ids = range(1, len(test_pred) + 1)

print(test_pred.shape)
# save submission file
prediction = pd.DataFrame({'id': ids, 'imdb_score_binned': test_pred})
prediction.to_csv("../Kaggle_submissions/dense_nn_notext.csv", index=False)

ValueError: in user code:

    File "C:\Users\Steven\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\engine\training.py", line 2440, in predict_function  *
        return step_function(self, iterator)
    File "C:\Users\Steven\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\engine\training.py", line 2425, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "C:\Users\Steven\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\engine\training.py", line 2413, in run_step  **
        outputs = model.predict_step(data)
    File "C:\Users\Steven\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\engine\training.py", line 2381, in predict_step
        return self(x, training=False)
    File "C:\Users\Steven\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\utils\traceback_utils.py", line 70, in error_handler
        raise e.with_traceback(filtered_tb) fro